# 일곱 가지 stream mode

같은 그래프에서 일곱 모드는 어떤 사건을 서로 다른 크기로 보여 줄까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 그래프 준비


In [2]:
# 두 노드가 상태 변경분과 사용자 정의 진행률을 내보내는 그래프를 만듭니다.
from typing import TypedDict
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    topic: str
    draft: str

def prepare(state: State) -> dict[str, str]:
    get_stream_writer()({"stage": "prepare", "progress": 50})
    return {"topic": state["topic"].strip()}

def write(state: State) -> dict[str, str]:
    get_stream_writer()({"stage": "write", "progress": 100})
    return {"draft": f"{state['topic']} 스트리밍 요약"}

builder = StateGraph(State)
builder.add_node("prepare", prepare)
builder.add_node("write", write)
builder.add_edge(START, "prepare")
builder.add_edge("prepare", "write")
builder.add_edge("write", END)
graph = builder.compile(checkpointer=InMemorySaver())


## 일곱 모드 실행


In [3]:
# 같은 입력을 일곱 모드로 실행하고 사건 수와 첫 자료형을 기록합니다.
modes = ["values", "updates", "messages", "custom", "checkpoints", "tasks", "debug"]
counts = {}
for index, mode in enumerate(modes):
    config = {"configurable": {"thread_id": f"mode-{index}"}}
    events = list(graph.stream({"topic": "고객 문의", "draft": ""}, config, stream_mode=mode))
    counts[mode] = len(events)
    sample_type = type(events[0]).__name__ if events else "없음"
    print(f"{mode:11} events={len(events):2} first={sample_type}")
assert counts["messages"] == 0
assert counts["custom"] == 2


values      events= 3 first=dict
updates     events= 2 first=dict
messages    events= 0 first=없음
custom      events= 2 first=dict
checkpoints events= 4 first=dict
tasks       events= 4 first=dict
debug       events= 8 first=dict


## 여러 모드를 한 번에 구독


In [4]:
# updates와 custom을 함께 구독해 모드 이름과 자료를 나란히 확인합니다.
config = {"configurable": {"thread_id": "multi"}}
events = list(graph.stream(
    {"topic": "배송", "draft": ""}, config,
    stream_mode=["updates", "custom"],
))
for mode, data in events:
    print(mode, data)
assert {mode for mode, _ in events} == {"updates", "custom"}


custom {'stage': 'prepare', 'progress': 50}
updates {'prepare': {'topic': '배송'}}
custom {'stage': 'write', 'progress': 100}
updates {'write': {'draft': '배송 스트리밍 요약'}}
